# SRA-Experiment „Läsion“ (Synapsenzerstörung).

In dieser Demo führen wir ein etwas Hacker-ähnliches Experiment durch, um zu beweisen, dass das trainierte Netzwerk von SRA „vollständig nach Funktion modularisiert“ ist.

1. Trainieren Sie das Modell gleichzeitig für `copy`- und `reverse`-Aufgaben.
2. **Zerstören Sie absichtlich die Gewichte eines bestimmten Experten (Synapse)** (setzen Sie sie auf Null)**, die häufig in der `reverse`-Aufgabe verwendet werden.
3. Bestätigen Sie, dass nach der Zerstörung die Aufgabe`reverse`nicht mehr gelöst werden kann, **die Aufgabe`copy`jedoch völlig unberührt bleibt (behält 100 % Genauigkeit bei)**.

## 1. Umgebungseinrichtung und Modellvorbereitung

In [1]:
import sys
if 'google.colab' in sys.modules:
    !git clone https://github.com/JunSuzukiJapan/SynapticRouter.git
    %cd SynapticRouter
    !pip install torch matplotlib seaborn

sys.path.append('.')
sys.path.append('./src')
if 'google.colab' not in sys.modules:
    sys.path.append('..')
    sys.path.append('../src')

import torch
torch.manual_seed(42)
import torch.nn.functional as F
torch.manual_seed(42)
from src.sra_gpu_models import MoESRAModel
class MoESRAConfig:
    def __init__(self, **kwargs):
        for k, v in kwargs.items():
            setattr(self, k, v)
from src.sra_experiment import make_multitask_batch, make_batch, make_optimizer, load_balance_loss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

config = MoESRAConfig(
    vocab_size=30,
    d_model=128,
    n_layers=2,
    n_heads=4,
    num_synapses=8,
    k=2,
    max_seq_len=32
)
model = MoESRAModel(config.vocab_size, config.d_model, config.n_layers, config.num_synapses, config.k, syn_hidden=256).to(device)
optimizer = make_optimizer(model, lr=0.001)

Using device: cpu


## 2. Führen Sie Multitasking-Lernen durch

In [2]:
print("Training started... (Copy & Reverse)")
model.train()
epochs = 1500
batch_size = 32
tasks = ["copy", "reverse"]

for epoch in range(epochs):
    x, y, _ = make_multitask_batch(tasks, batch_size, min_len=4, max_len=8, device=device)
    
    optimizer.zero_grad()
    y_in = torch.cat([torch.full((y.size(0), 1), 1, dtype=torch.long, device=device), y[:, :-1]], dim=1)
    outputs, routing_weights, _ = model(x, y_in)
    
    loss = F.cross_entropy(outputs.reshape(-1, config.vocab_size), y.reshape(-1))
    lb_loss = load_balance_loss(routing_weights) * 0.1
    total_loss = loss + lb_loss
    
    total_loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f}")

print("Training finished!")

Training started... (Copy & Reverse)


Epoch 100/1500 | Loss: 1.3509


Epoch 200/1500 | Loss: 0.8547


Epoch 300/1500 | Loss: 0.8108


Epoch 400/1500 | Loss: 0.5746


...

Epoch 1100/1500 | Loss: 0.0925


Epoch 1200/1500 | Loss: 0.1350


Epoch 1300/1500 | Loss: 0.1062


Epoch 1400/1500 | Loss: 0.0142


Epoch 1500/1500 | Loss: 0.0166
Training finished!


## 3. Leistungstest vor der Läsion und Identifizierung der Zielsynapse
Bestätigen Sie, dass beide Aufgaben gelöst werden können, und suchen Sie die in der `reverse`-Aufgabe am häufigsten verwendete Synapse (das Ziel).

In [3]:
def test_task(task_name):
    model.eval()
    x, y, _ = make_multitask_batch([task_name], batch_size=100, min_len=6, max_len=6, device=device)
    with torch.no_grad():
        y_in = torch.cat([torch.full((y.size(0), 1), 1, dtype=torch.long, device=device), y[:, :-1]], dim=1)
        outputs, routing_weights, _ = model(x, y_in)
        preds = outputs.argmax(dim=-1)
        
    # Percentage of correct answers excluding padding
    mask = y != 0 # 0 is PAD
    acc = (preds[mask] == y[mask]).float().mean().item()
    
    # Aggregate used synapses (Layer 0)
    chosen = routing_weights[0].argmax(dim=-1)
    usage = torch.bincount(chosen.flatten(), minlength=config.num_synapses)
    
    print(f"[{task_name.upper()}] Accuracy: {acc*100:.1f}%")
    return usage

print("=== Before Lesion ===")
copy_usage = test_task("copy")
reverse_usage = test_task("reverse")

# Target synapses that are more active in reverse than in copy
diff = reverse_usage - copy_usage
target_synapses = [i for i, val in enumerate(diff) if val > 0]
print(f"\n=> Target for destruction: Synapses {target_synapses} (highly active in REVERSE)")
# Set the synapse most used in Reverse task as 'destruction target'
target_synapses = [i for i, val in enumerate(diff) if val > 0]
print(f"\n=> Target for destruction: Synapse {target_synapses} (highly active in REVERSE)")

=== Before Lesion ===
[COPY] Accuracy: 100.0%
[REVERSE] Accuracy: 96.0%

=> Target for destruction: Synapses [2] (highly active in REVERSE)

=> Target for destruction: Synapse [2] (highly active in REVERSE)


## 4. Synapsenzerstörung (Läsion)
Überschreiben Sie absichtlich die Gewichte der Zielsynapse mit Null.

In [4]:
print(f"Destroying Synapses {target_synapses} in all layers...")

with torch.no_grad():
    # Set specific expert weights in MoESRABlock to 0
    for layer_to_destroy in range(config.n_layers):
        block = model.blocks[layer_to_destroy]
        for target_synapse in target_synapses:
            block.w1.data[target_synapse].zero_()
            block.b1.data[target_synapse].zero_()
            block.w2.data[target_synapse].zero_()
            block.b2.data[target_synapse].zero_()

print("Destruction complete. 💥")

Destroying Synapses [2] in all layers...
Destruction complete. 💥


## 5. Leistungstest nach der Läsion
Erneuter Test mit zerstörtem Teil des Gehirns (Synapse).
Die Genauigkeit von`reverse`wird sinken, aber`copy`, das andere Synapsen verwenden sollte, bleibt unversehrt!

In [5]:
print("=== After Lesion ===")
test_task("copy")
test_task("reverse")

print("\n💡 [Insight]")
print("If you destroy a part of a large monolithic neural network (like a standard Transformer) with zeros, usually all tasks are devastated.")
print("However, in SRA, because independent expert pathways (synapses) are formed for each task,")
print("it demonstrates astonishing robustness: 'Even if the brain for Reverse breaks, the brain for Copy continues to work flawlessly'!")

=== After Lesion ===
[COPY] Accuracy: 99.4%


[REVERSE] Accuracy: 86.1%

💡 [Insight]
If you destroy a part of a large monolithic neural network (like a standard Transformer) with zeros, usually all tasks are devastated.
However, in SRA, because independent expert pathways (synapses) are formed for each task,
it demonstrates astonishing robustness: 'Even if the brain for Reverse breaks, the brain for Copy continues to work flawlessly'!
